In [1]:
# !pip install requests pandas sentence-transformers hdbscan google-generativeai jupyter

In [7]:
!pip install streamlit requests sentence-transformers hdbscan pandas numpy google-genai

In [ ]:
import requests
import pandas as pd
import os

# --- Credentials ---
BSKY_HANDLE = os.getenv("BSKY_HANDLE")
BSKY_APP_PASSWORD = os.getenv("BSKY_APP_PASSWORD")

def fetch_bluesky_posts(query, target_count=1000):
    print(f"Authenticating as {BSKY_HANDLE}...")
    
    # 1. Create a session to get the auth token
    session_url = "https://bsky.social/xrpc/com.atproto.server.createSession"
    session_data = {"identifier": BSKY_HANDLE, "password": BSKY_APP_PASSWORD}
    session_resp = requests.post(session_url, json=session_data).json()
    
    if "accessJwt" not in session_resp:
        raise Exception(f"Failed to authenticate: {session_resp}")
        
    auth_token = session_resp["accessJwt"]
    headers = {"Authorization": f"Bearer {auth_token}"}
    
    # 2. Search for posts iteratively
    search_url = "https://bsky.social/xrpc/app.bsky.feed.searchPosts"
    
    posts_data = []
    cursor = None
    
    print(f"Fetching {target_count} posts for '{query}'...")
    while len(posts_data) < target_count:
        params = {"q": query, "limit": 100} # 100 is the max per request
        if cursor:
            params["cursor"] = cursor
            
        resp = requests.get(search_url, headers=headers, params=params).json()
        new_posts = resp.get("posts", [])
        
        if not new_posts:
            break # No more posts available
            
        for post in new_posts:
            # We extract the text, the timestamp, and the author
            posts_data.append({
                "text": post["record"]["text"],
                "created_at": post["record"]["createdAt"],
                "author": post["author"]["handle"]
            })
            
        cursor = resp.get("cursor")
        if not cursor:
            break
            
    # Keep only the target amount and convert to a DataFrame
    df = pd.DataFrame(posts_data[:target_count])
    print(f"Successfully fetched {len(df)} posts.")
    return df

# Test
# df_posts = fetch_bluesky_posts("skincare", target_count=1000)
# df_posts.head()

Authenticating as alosantillan.bsky.social...
Fetching 1000 posts for 'skincare'...
Successfully fetched 1000 posts.


,text,created_at,author
0,Experts explain how vegan PDRN compares to sal...,2026-07-27T19:40:12Z,sacbee.com
1,Thinking about trying spicule skincare? Learn ...,2026-07-27T19:30:31Z,sacbee.com
2,Os lançamentos de beleza de julho de 2026 que ...,2026-07-27T19:22:19.000Z,flipboardbr.flipboard.com.ap.brid.gy
3,skincare setup.,2026-07-27T19:02:31.612191Z,420formulator.bsky.social
4,Want to know why cocoa butter is a favourite i...,2026-07-27T19:02:30.534635Z,420formulator.bsky.social
